# LCMFreq — Frequent Itemset Mining Demo

**Thuật toán:** LCMFreq (Uno, Kiyomi & Arimura, FIMI 2004)  
**Ngôn ngữ:** Julia 1.12 · **Môn học:** CSC14004 — Khai thác Dữ liệu và Ứng dụng

Notebook này trình bày:
1. Khái niệm TID-set, support, tính chất anti-monotone
2. Trace từng bước trên CSDL đồ chơi (Chương 2 báo cáo)
3. So sánh Baseline vs Optimised — correctness & speedup
4. Ứng dụng: sinh association rules từ frequent itemsets

## 0 · Setup

In [1]:
import Pkg

# Project root = parent of notebooks/
PROJ = normpath(joinpath(@__DIR__, ".."))
Pkg.activate(PROJ)
Pkg.instantiate()

include(joinpath(PROJ, "src", "io", "reader.jl"))
include(joinpath(PROJ, "src", "structures.jl"))

# Load each implementation in its own module to avoid name clashes
module Base_
    import Main: PROJ
    include(joinpath(PROJ, "src", "algorithm", "lcmfreq_base.jl"))
end

module Opt_
    import Main: PROJ
    include(joinpath(PROJ, "src", "algorithm", "lcmfreq.jl"))
end

println("Julia $(VERSION) — LCMFreq loaded")
println("  Baseline : Base_.lcmfreq_base")
println("  Optimised: Opt_.lcmfreq")

  Activating project at `C:\Users\admin\Downloads\FrequentItemsetMining_Lab2`


Julia 1.12.6 — LCMFreq loaded
  Baseline : Base_.lcmfreq_base
  Optimised: Opt_.lcmfreq


## 1 · CSDL đồ chơi — Ví dụ 1 (Chương 2 báo cáo)

$\mathcal{I} = \{A,B,C,D\}$, $\sigma = 2$ (tuyệt đối)

| TID | Items       |
|-----|-------------|
| T1  | A, B, C, D  |
| T2  | A, B, D     |
| T3  | A, B, C     |
| T4  | B, C        |
| T5  | A, B, C, D  |

In [2]:
# Item encoding: A=1, B=2, C=3, D=4
toy_db = [
    [1, 2, 3, 4],   # T1: A B C D
    [1, 2, 4],      # T2: A B D
    [1, 2, 3],      # T3: A B C
    [2, 3],         # T4: B C
    [1, 2, 3, 4],   # T5: A B C D
]
name   = Dict(1=>"A", 2=>"B", 3=>"C", 4=>"D")
minsup = 2

result = Opt_.lcmfreq(toy_db, minsup)

println("Found $(length(result.itemsets)) frequent itemsets (sigma=$minsup):\n")
for fi in sort(result.itemsets, by=x->(length(x.items), x.items))
    println("  {$(join([name[i] for i in fi.items], ","))}  sup=$(fi.support)")
end

Found 15 frequent itemsets (sigma=2):

  {A}  sup=4


  {B}  sup=5
  {C}  sup=4
  {D}  sup=3
  {A,B}  sup=4
  {A,C}  sup=3
  {A,D}  sup=3
  {B,C}  sup=4
  {B,D}  sup=3
  {C,D}  sup=2
  {A,B,C}  sup=3
  {A,B,D}  sup=3
  {A,C,D}  sup=2
  {B,C,D}  sup=2
  {A,B,C,D}  sup=2


**Kỳ vọng:** 15 frequent itemsets — tất cả $2^4 - 1 = 15$ tập con khác rỗng đều có $\text{sup} \geq 2$.

## 2 · TID-set và phép giao BitArray

LCMFreq dùng **vertical data format**: mỗi item lưu TID-set — danh sách transaction chứa nó.  
Support của itemset = kích thước giao các TID-set thành phần.

In [3]:
n = length(toy_db)

tidset = Dict{Int,BitArray{1}}(item => falses(n) for item in 1:4)
for (tid, tx) in enumerate(toy_db)
    for item in tx
        tidset[item][tid] = true
    end
end

println("TID-sets (n=$n transactions):\n")
for item in 1:4
    tids = findall(tidset[item])
    println("  T($(name[item])) = $tids   sup=$(length(tids))")
end

println("\nGiao bằng AND bit (T(X) = AND cac TID-set thanh phan):")
ab = tidset[1] .& tidset[2]
println("  T({A,B})   = $(findall(ab))  sup=$(count(ab))")
println("  T({A,C})   = $(findall(tidset[1] .& tidset[3]))  sup=$(count(tidset[1] .& tidset[3]))")
println("  T({B,C,D}) = $(findall(tidset[2] .& tidset[3] .& tidset[4]))  sup=$(count(tidset[2] .& tidset[3] .& tidset[4]))")

TID-sets (n=5 transactions):



  T(A) = [1, 2, 3, 5]   sup=4
  T(B) = [1, 2, 3, 4, 5]   sup=5
  T(C) = [1, 3, 4, 5]   sup=4
  T(D) = [1, 2, 5]   sup=3

Giao bằng AND bit (T(X) = AND cac TID-set thanh phan):
  T({A,B})   = [1, 2, 3, 5]  sup=4
  T({A,C})   = [1, 3, 5]  sup=3
  T({B,C,D}) = [1, 5]  sup=2

## 3 · Baseline vs Optimised — Kiểm tra tính đúng đắn

In [4]:
t_base = @elapsed r_base = Base_.lcmfreq_base(toy_db, minsup)
t_opt  = @elapsed r_opt  = Opt_.lcmfreq(toy_db, minsup)

key(fi) = (sort(fi.items), fi.support)
set_base = Set(key(fi) for fi in r_base.itemsets)
set_opt  = Set(key(fi) for fi in r_opt.itemsets)

println("Baseline   : $(length(r_base.itemsets)) itemsets  $(round(t_base*1000, digits=2)) ms")
println("Optimised  : $(length(r_opt.itemsets)) itemsets   $(round(t_opt*1000, digits=2)) ms")
println("Ket qua khop: $(set_base == set_opt ? "YES" : "NO")")

Baseline   : 15 itemsets  238.6 ms
Optimised  : 15 itemsets   0.03 ms
Ket qua khop: YES


## 4 · Benchmark trên Chess (dense dataset)

Chess: 3,196 transactions, 75 items, avg length 37.

In [5]:
chess_path = joinpath(PROJ, "data", "benchmark", "chess.dat")

if isfile(chess_path)
    txs = read_transactions(chess_path)
    ms  = round(Int, 0.80 * length(txs))

    println("Chess: $(length(txs)) transactions | minsup=$ms (80%)\n")

    t_base = @elapsed r_b = Base_.lcmfreq_base(txs, ms)
    t_opt  = @elapsed r_o = Opt_.lcmfreq(txs, ms)

    println("  Baseline  : $(length(r_b.itemsets)) itemsets  $(round(t_base*1000, digits=1)) ms")
    println("  Optimised : $(length(r_o.itemsets)) itemsets  $(round(t_opt*1000, digits=1)) ms")
    println("  Speedup   : $(round(t_base/t_opt, digits=1))x")
else
    println("chess.dat not found -- place in data/benchmark/")
end

Chess: 3196 transactions | minsup=2557 (80%)


  Baseline  : 8227 itemsets  1518.6 ms
  Optimised : 8227 itemsets  7.7 ms
  Speedup   : 196.8x


## 5 · BitArray AND vs Vector{Int} merge — Micro-benchmark

In [6]:
using BenchmarkTools, Random

N = 10_000
rng = MersenneTwister(42)
density = 0.7

v_A = sort(filter(_ -> rand(rng) < density, 1:N))
v_B = sort(filter(_ -> rand(rng) < density, 1:N))
b_A = falses(N); b_A[v_A] .= true
b_B = falses(N); b_B[v_B] .= true

function intersect_sorted(a, b)
    res = Int[]; i = j = 1
    while i <= length(a) && j <= length(b)
        if a[i] == b[j]; push!(res, a[i]); i += 1; j += 1
        elseif a[i] < b[j]; i += 1 else j += 1 end
    end
    return res
end

t_vec = @belapsed intersect_sorted($v_A, $v_B)
t_bit = @belapsed ($b_A .& $b_B)

println("N=$N, density=$(round(Int, density*100))%")
println("Vector{Int} merge : $(round(t_vec*1e6, digits=1)) us")
println("BitArray AND      : $(round(t_bit*1e6, digits=1)) us")
println("Speedup           : $(round(t_vec/t_bit, digits=1))x")

N=10000, density=70%
Vector{Int} merge : 33.8 us
BitArray AND      : 0.1 us
Speedup           : 616.5x


## 6 · Association Rules từ Frequent Itemsets

In [7]:
sup     = Dict(sort(fi.items) => fi.support for fi in r_opt.itemsets)
N_tx    = length(toy_db)
minconf = 0.70

println("Association rules (minconf=$(round(Int, minconf*100))%):\n")
for fi in sort(r_opt.itemsets, by=x->x.items)
    length(fi.items) < 2 && continue
    items = sort(fi.items)
    for ci in 1:length(items)
        ant     = [items[i] for i in 1:length(items) if i != ci]
        sup_ant = get(sup, sort(ant), 0)
        sup_ant == 0 && continue
        conf = fi.support / sup_ant
        if conf >= minconf
            println("  {$(join([name[i] for i in ant], ","))} => {$(name[items[ci]])}" *
                    "  conf=$(round(conf*100, digits=0))%  sup=$(fi.support)")
        end
    end
end

Association rules (minconf=70%):



  {B} => {A}  conf=80.0%  sup=4
  {A} => {B}  conf=100.0%  sup=4
  {B,C} => {A}  conf=75.0%  sup=3
  {A,C} => {B}  conf=100.0%  sup=3
  {A,B} => {C}  conf=75.0%  sup=3
  {B,C,D} => {A}  conf=100.0%  sup=2
  {A,C,D} => {B}  conf=100.0%  sup=2
  {B,D} => {A}  conf=100.0%  sup=3
  {A,D} => {B}  conf=100.0%  sup=3
  {A,B} => {D}  conf=75.0%  sup=3
  {C} => {A}  conf=75.0%  sup=3
  {A} => {C}  conf=75.0%  sup=3
  {C,D} => {A}  conf=100.0%  sup=2
  {D} => {A}  conf=100.0%  sup=3
  {A} => {D}  conf=75.0%  sup=3
  {C} => {B}  conf=100.0%  sup=4
  {B} => {C}  conf=80.0%  sup=4
  {C,D} => {B}  conf=100.0%  sup=2
  {D} => {B}  conf=100.0%  sup=3


## Tóm tắt kết quả thực nghiệm đầy đủ

| Dataset | Speedup trung vị | Max speedup |
|---------|-----------------|-------------|
| Chess | 135× | 241× |
| Mushroom | 58× | 101× |
| Connect | 6× | 34× |
| Pumsb | 343× | 343× |
| Accidents | 7× | 22× |
| Retail | 2× | 6× |
| T10I4D100K | 8× | 9× |
| T40I10D100K | 32× | 32× |

Xem chi tiết: `docs/Report.pdf` · Chạy đầy đủ: `scripts/run_experiments.jl`